<div style="background: linear-gradient(135deg, #0A0F1D 0%, #121A2F 100%); border: 1px solid #00E676; border-radius: 12px; padding: 30px; color: #FFFFFF; font-family: 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0, 0, 0, 0.4);">
<div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid rgba(0, 230, 118, 0.2); padding-bottom: 12px; margin-bottom: 20px;">
<span style="font-size: 12px; letter-spacing: 2px; color: #00E676; text-transform: uppercase; font-weight: 700;">SYSTEM PIPELINE • BEHAVIORAL TELEMETRY</span>
<span style="background: rgba(0, 230, 118, 0.1); border: 1px solid #00E676; color: #00E676; padding: 3px 12px; border-radius: 20px; font-size: 11px; font-weight: 600;">150,000 SAMPLES</span>
</div>
<h1 style="color: #FFFFFF; margin: 0 0 8px 0; font-size: 26px; font-weight: 800; letter-spacing: -0.5px;">Human Digital Behavior & Wellbeing Analytics</h1>
<p style="color: #90A4AE; font-size: 13px; margin: 0 0 25px 0; max-width: 800px; line-height: 1.5;">High-accuracy gradient boosting architecture paired with interactive Plotly diagnostics to evaluate screen time fatigue, sleep metrics, and cognitive wellbeing impacts.</p>
<div style="background: rgba(10, 15, 29, 0.8); border: 1px solid rgba(255, 255, 255, 0.08); border-radius: 8px; padding: 20px;">
<h3 style="color: #00E676; margin: 0 0 15px 0; font-size: 15px; font-weight: 700; text-transform: uppercase; letter-spacing: 1px;">Table of Contents</h3>
<div style="display: flex; flex-direction: column; gap: 12px; font-size: 14px; font-weight: 600;">
<div style="color: #ECEFF1; display: flex; align-items: center;"><span style="color: #00E676; font-weight: 800; width: 28px;">1.</span><span>DATA LOADING</span></div>
<div style="color: #ECEFF1; display: flex; align-items: center;"><span style="color: #00E676; font-weight: 800; width: 28px;">2.</span><span>FEATURE ENGINEERING & TARGET CLEANING</span></div>
<div style="color: #ECEFF1; display: flex; align-items: center;"><span style="color: #00E676; font-weight: 800; width: 28px;">3.</span><span>INTERACTIVE 3D EXPLORATORY SURFACING</span></div>
<div style="color: #ECEFF1; display: flex; align-items: center;"><span style="color: #00E676; font-weight: 800; width: 28px;">4.</span><span>HIGH-ACCURACY SPEED MODELING & BENCHMARKING</span></div>
<div style="color: #ECEFF1; display: flex; align-items: center;"><span style="color: #00E676; font-weight: 800; width: 28px;">5.</span><span>INTERACTIVE DIAGNOSTIC CHARTS & FEATURE IMPORTANCE</span></div>
</div>
</div>

## 1. DATA LOADING 

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# -------------------------------------------------------------------------
# 1. DATA LOADING & DYNAMIC PATH DETECTION
# -------------------------------------------------------------------------
paths_to_check = [
    '/kaggle/input/human-digital-behavior-and-wellbeing-dataset-150k/human_digital_behavior_dataset.csv',
    '/kaggle/input/human-digital-behavior-and-wellbeing-dataset-150k/human_digital_behavior_and_wellbeing_150k.csv',
    '/kaggle/input/datasets/mobeenfatimah/human-digital-behavior-and-wellbeing-dataset-150k/human_digital_behavior_dataset.csv'
    '/kaggle/input/datasets/mobeenfatimah/human-digital-behavior-and-wellbeing-dataset-150k/human_digital_behavior_wellbeing.csv'
]

df = None
for path in paths_to_check:
    try:
        df = pd.read_csv(path)
        print(f"Successfully loaded dataset from: {path}")
        break
    except FileNotFoundError:
        continue

if df is None:
    print("Kaggle path not detected. Initializing synthetic 150,000 behavioral telemetry dataset...")
    np.random.seed(42)
    n = 150000
    
    screen_time_hrs = np.random.uniform(1.0, 14.0, size=n)
    social_media_hrs = screen_time_hrs * np.random.uniform(0.2, 0.75, size=n)
    notifications_per_day = np.random.randint(20, 350, size=n)
    late_night_usage_min = np.random.uniform(0, 240, size=n)
    physical_activity_min = np.random.uniform(0, 120, size=n)
    sleep_hours = 8.5 - (late_night_usage_min / 60.0) * 0.6 + np.random.normal(0, 0.5, size=n)
    sleep_hours = np.clip(sleep_hours, 3.5, 10.0)
    
    wellbeing = 90.0 - (screen_time_hrs * 2.2) - (late_night_usage_min * 0.08) - (notifications_per_day * 0.05) + (sleep_hours * 3.5) + (physical_activity_min * 0.12)
    wellbeing = np.clip(wellbeing + np.random.normal(0, 3.0, size=n), 10.0, 100.0)
    
    df = pd.DataFrame({
        'user_id': [f"USR-{i:06d}" for i in range(1, n + 1)],
        'screen_time_hours': screen_time_hrs,
        'social_media_hours': social_media_hrs,
        'notifications_per_day': notifications_per_day,
        'late_night_usage_minutes': late_night_usage_min,
        'sleep_hours': sleep_hours,
        'physical_activity_minutes': physical_activity_min,
        'device_type': np.random.choice(['Mobile', 'Desktop', 'Tablet'], size=n),
        'wellbeing_score': wellbeing
    })

print(f"Dataset Dimensions: {df.shape[0]:,} Rows | {df.shape[1]} Columns\n")


Kaggle path not detected. Initializing synthetic 150,000 behavioral telemetry dataset...
Dataset Dimensions: 150,000 Rows | 9 Columns



## 2. FEATURE ENGINEERING & TARGET CLEANING

In [2]:
def engineer_wellbeing_features(data):
    df_feat = data.copy()
    
    target_matches = [c for c in df_feat.columns if any(k in c.lower() for k in ['wellbeing', 'stress', 'mental', 'score', 'health'])]
    TARGET_COL = target_matches[0] if target_matches else df_feat.select_dtypes(include=[np.number]).columns[-1]
    
    def find_col(keys):
        for k in keys:
            for c in df_feat.columns:
                if k in c.lower() and c != TARGET_COL:
                    return c
        return None

    screen_col = find_col(['screen_time', 'daily_usage', 'screen'])
    sleep_col = find_col(['sleep', 'sleep_duration', 'bedtime'])
    night_col = find_col(['late_night', 'night_usage', 'night'])
    notif_col = find_col(['notification', 'alerts', 'notif'])

    # High-impact Non-linear Ratios
    if screen_col and sleep_col:
        df_feat['screen_to_sleep_ratio'] = df_feat[screen_col] / (df_feat[sleep_col] + 0.1)
    if notif_col and screen_col:
        df_feat['notifications_per_hour'] = df_feat[notif_col] / (df_feat[screen_col] + 0.1)
    if night_col and sleep_col:
        df_feat['circadian_disruption_index'] = (df_feat[night_col] / 60.0) / (df_feat[sleep_col] + 0.1)

    return df_feat, TARGET_COL, screen_col, sleep_col

df_engineered, TARGET_COL, SCREEN_COL, SLEEP_COL = engineer_wellbeing_features(df)
df_engineered = df_engineered.dropna(subset=[TARGET_COL])


## 3. INTERACTIVE 3D EXPLORATORY SURFACING

In [3]:
sample_3d = df_engineered.sample(n=min(3500, len(df_engineered)), random_state=42)

x_axis = SCREEN_COL if SCREEN_COL in sample_3d.columns else sample_3d.columns[1]
y_axis = SLEEP_COL if SLEEP_COL in sample_3d.columns else sample_3d.columns[2]

fig_3d = px.scatter_3d(
    sample_3d,
    x=x_axis,
    y=y_axis,
    z=TARGET_COL,
    color=TARGET_COL,
    color_continuous_scale='Tealrose',
    title=f'<b>3D Behavioral Surface</b> ({x_axis} vs {y_axis} vs {TARGET_COL})',
    opacity=0.75
)

fig_3d.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0A0F1D',
    plot_bgcolor='#0A0F1D',
    height=550,
    margin=dict(l=10, r=10, b=10, t=40)
)

# Forces Kaggle to embed the output into the notebook HTML state for all visitors
fig_3d.show(renderer="iframe")

## 4. HIGH-ACCURACY SPEED MODELING & BENCHMARKING

In [4]:
id_cols = [c for c in df_engineered.columns if 'id' in c.lower()]
df_model = df_engineered.drop(columns=id_cols, errors='ignore')

X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# HistGradientBoosting optimized for max throughput on 150k rows
model = HistGradientBoostingRegressor(
    categorical_features=categorical_cols if len(categorical_cols) > 0 else None,
    max_iter=350,
    learning_rate=0.05,
    max_depth=8,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Model Performance Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("HIGH-ACCURACY MODEL BENCHMARK RESULTS")
print("=" * 60)
print(f"R-Squared Score (Variance Explained) : {r2 * 100:.2f}%")
print(f"Root Mean Squared Error (RMSE)      : {rmse:.4f}")
print(f"Mean Absolute Error (MAE)            : {mae:.4f}")
print("=" * 60 + "\n")

HIGH-ACCURACY MODEL BENCHMARK RESULTS
R-Squared Score (Variance Explained) : 94.28%
Root Mean Squared Error (RMSE)      : 2.7685
Mean Absolute Error (MAE)            : 2.0993



## 5. INTERACTIVE DIAGNOSTIC CHARTS & FEATURE IMPORTANCE

In [5]:
# Permutation Feature Importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=3, random_state=42, n_jobs=-1)
sorted_idx = perm_importance.importances_mean.argsort()[-10:]

imp_df = pd.DataFrame({
    'Feature': X_test.columns[sorted_idx],
    'Importance': perm_importance.importances_mean[sorted_idx]
})

fig_imp = px.bar(
    imp_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title="<b>Top Feature Drivers (Permutation Impact)</b>",
    color='Importance',
    color_continuous_scale='Viridis',
    template="plotly_dark"
)

fig_imp.update_layout(paper_bgcolor='#0A0F1D', plot_bgcolor='#121A2F', height=400, showlegend=False)

# Forces Kaggle to embed the output directly into the notebook HTML for public viewers
fig_imp.show(renderer="iframe")

In [6]:
# Actual vs Predicted Residual Subplots
eval_df = pd.DataFrame({
    'Actual': y_test, 
    'Predicted': y_pred, 
    'Residual': y_test - y_pred
}).sample(min(2000, len(y_test)), random_state=42)

fig_eval = make_subplots(
    rows=1, 
    cols=2, 
    subplot_titles=('<b>Actual vs Predicted</b>', '<b>Error Residuals</b>')
)

fig_eval.add_trace(
    go.Scatter(
        x=eval_df['Actual'], 
        y=eval_df['Predicted'], 
        mode='markers', 
        marker=dict(color='#00E676', opacity=0.4, size=4)
    ), 
    row=1, col=1
)

fig_eval.add_trace(
    go.Scatter(
        x=[y_test.min(), y_test.max()], 
        y=[y_test.min(), y_test.max()], 
        mode='lines', 
        line=dict(color='#FF1744', dash='dash')
    ), 
    row=1, col=1
)

fig_eval.add_trace(
    go.Histogram(
        x=eval_df['Residual'], 
        nbinsx=40, 
        marker_color='#00E5FF', 
        opacity=0.75
    ), 
    row=1, col=2
)

fig_eval.update_layout(
    template='plotly_dark', 
    paper_bgcolor='#0A0F1D', 
    plot_bgcolor='#121A2F', 
    height=400, 
    showlegend=False
)

# Forces Kaggle to embed the output directly into the notebook HTML for public viewers
fig_eval.show(renderer="iframe")